In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

project_root = Path("/Users/miki_przygoda/Projects/GitHub-Projects/Maths Data Script")
csv_path = project_root / "full_dataset" / "full_dataset_20260313_134447_4ccad990.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

print(f"Reading CSV: {csv_path}")
df = pd.read_csv(csv_path)

# Filtered data (without Mikolaj) - case insensitive
df_filtered = df[df["Member_Name"].astype(str).str.strip().str.lower() != "mikolaj"].copy()

# Columns to use in the heatmap
vars_to_corr = [
    "CPU_Usage_Pct",
    "RAM_Free_MB",
    "Clock_Speed_GHz",
    "Execution_Time_ms",
    "Power_Plugged",
    "Process_Count",
]

# Check required columns
missing = [col for col in vars_to_corr + ["Member_Name"] if col not in df.columns]
if missing:
    raise KeyError(f"Missing columns in CSV: {missing}")

def prepare_numeric_data(data, columns):
    prepared = data[columns].copy()

    # Normalize common boolean/string forms in Power_Plugged
    if "Power_Plugged" in prepared.columns:
        prepared["Power_Plugged"] = (
            prepared["Power_Plugged"]
            .replace({
                True: 1, False: 0,
                "True": 1, "False": 0,
                "true": 1, "false": 0,
                "Yes": 1, "No": 0,
                "yes": 1, "no": 0,
                "Y": 1, "N": 0,
                "y": 1, "n": 0,
                "Plugged": 1, "Unplugged": 0,
                "plugged": 1, "unplugged": 0
            })
        )

    # Coerce everything to numeric
    for col in columns:
        prepared[col] = pd.to_numeric(prepared[col], errors="coerce")

    return prepared

def create_staircase_heatmap(data, title, output_path):
    numeric_data = prepare_numeric_data(data, vars_to_corr)
    corr = numeric_data.corr()

    # Mask upper triangle including diagonal
    mask = np.triu(np.ones_like(corr, dtype=bool), k=0)

    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".4f",
        cmap="coolwarm",
        center=0,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 10}
    )

    # Add "-" on the diagonal
    for i in range(len(corr)):
        ax.text(
            i + 0.5, i + 0.5, "-",
            ha="center", va="center",
            color="black", fontsize=14, fontweight="bold"
        )

    plt.title(title, fontsize=15)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()

# Output files
full_heatmap_path = project_root / "staircase_heatmap_full.png"
filtered_heatmap_path = project_root / "staircase_heatmap_filtered.png"

# Generate the heatmaps
create_staircase_heatmap(
    df,
    "Correlation Heatmap - Full Dataset (With Mikolaj)",
    full_heatmap_path
)
create_staircase_heatmap(
    df_filtered,
    "Correlation Heatmap - Filtered Dataset (Without Mikolaj)",
    filtered_heatmap_path
)

print(f"Saved: {full_heatmap_path}")
print(f"Saved: {filtered_heatmap_path}")

# Precise Regression for Model 3 (Filtered Data)
regression_data = df_filtered[["Process_Count", "CPU_Usage_Pct", "Execution_Time_ms"]].copy()
for col in regression_data.columns:
    regression_data[col] = pd.to_numeric(regression_data[col], errors="coerce")
regression_data = regression_data.dropna()

X = regression_data[["Process_Count", "CPU_Usage_Pct"]]
y = regression_data["Execution_Time_ms"]

reg = LinearRegression().fit(X, y)

print(f"Intercept: {reg.intercept_:.10f}")
print(f"Coeff Process_Count: {reg.coef_[0]:.10f}")
print(f"Coeff CPU_Usage_Pct: {reg.coef_[1]:.10f}")
print(f"R2 Score: {reg.score(X, y):.10f}")

Reading CSV: /Users/miki_przygoda/Projects/GitHub-Projects/Maths Data Script/full_dataset/full_dataset_20260313_134447_4ccad990.csv
Saved: /Users/miki_przygoda/Projects/GitHub-Projects/Maths Data Script/staircase_heatmap_full.png
Saved: /Users/miki_przygoda/Projects/GitHub-Projects/Maths Data Script/staircase_heatmap_filtered.png
Intercept: 42.9552989579
Coeff Process_Count: 0.1151608159
Coeff CPU_Usage_Pct: -0.0551914599
R2 Score: 0.5804168853
